In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt

from pre_processing import img_processing

In [ ]:
# load trained model
model = tf.keras.models.load_model(
    "sinhala_convnext_canny.keras"    #model name
)

# IMPORTANT: use SAME order as training
CLASS_NAMES = [
       
]

IMG_SIZE = 128
SEGMENTED_DIR = "segmented_letters"

In [ ]:
def preprocess_for_model(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise ValueError(f"Cannot read image: {img_path}")

    # ---- YOUR preprocessing ----
    img = img_processing(img)

    # resize to model input
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # normalize
    img = img.astype("float32") / 255.0

    # GRAY → RGB (IMPORTANT)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    # add batch dimension
    img = img.reshape(1, IMG_SIZE, IMG_SIZE, 3)

    return img

In [ ]:
def predict_letter(img_path):
    x = preprocess_for_model(img_path)

    preds = model.predict(x, verbose=0)[0]
    class_id = int(np.argmax(preds))
    confidence = float(preds[class_id])

    return CLASS_NAMES[class_id], confidence



In [ ]:
results = []

for file in sorted(os.listdir(SEGMENTED_DIR)):
    if not file.lower().endswith((".png", ".jpg", ".jpeg")):
        continue

    path = os.path.join(SEGMENTED_DIR, file)

    label, conf = predict_letter(path)

    results.append((file, label, conf))

    print(f"{file}  →  {label}  ({conf:.2f})")


In [ ]:
plt.figure(figsize=(12, 3))

for i, (file, label, conf) in enumerate(results):
    img = cv2.imread(
        os.path.join(SEGMENTED_DIR, file),
        cv2.IMREAD_GRAYSCALE
    )

    plt.subplot(1, len(results), i + 1)
    plt.imshow(img, cmap="gray")
    plt.title(label)
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Geting class names of the model
print("Model output neurons:", model.output_shape[-1])
print("Number of class names:", len(CLASS_NAMES))
print("CLASS_NAMES:", CLASS_NAMES)


In [ ]:
# predict using specific image
test_img = "segmented_letters/char_2.png"

label, conf = predict_letter(test_img)

img = cv2.imread(test_img, cv2.IMREAD_GRAYSCALE)

plt.imshow(img, cmap="gray")
plt.title(f"Predicted: {label} ({conf:.2f})")
plt.axis("off")
plt.show()
